In [ ]:
import os
import json
import pickle
from collections import defaultdict
from itertools import product

import numpy as np
import pandas as pd


from tqdm import tqdm
from xgboost import XGBRanker
from sentence_transformers import SentenceTransformer


In [2]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)


def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)


def ndcg_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0


def hit_score_at_k(rec_k, rel_set):
    cant_relevantes = set(rec_k).intersection(set(rel_set))
    return len(cant_relevantes)


def map_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0


def diversity_at_k(rec_k, info_videojuegos):
    generos_total = set()
    for app_id in rec_k:
        for genero in info_videojuegos[app_id]:
            generos_total.add(genero)
    if not generos_total:
        return 0
    return len(generos_total)


def f1_at_k(rec_k, rel_set):
    if len(rec_k) == 0 or len(rel_set) == 0:
        return 0.0

    p = precision_at_k(rec_k, rel_set)
    r = recall_at_k(rec_k, rel_set)

    if (p + r) <= 0:
        return 0.0

    return 2 * p * r / (p + r)


In [ ]:

base_dir = os.getcwd()
data_dir = os.path.join(base_dir, "..", "data", "split")

ruta_train = os.path.join("train_split.csv")
ruta_test = os.path.join("test_split.csv")
ruta_val = os.path.join("val_split.csv")

ruta_metadata = os.path.join("games_metadata.json")
ruta_imagenes_url = os.path.join("steam_media_data.csv")

train_set = pd.read_csv(ruta_train)
test_set = pd.read_csv(ruta_test)

train_set["hours"] = np.log1p(train_set["hours"])
test_set["hours"] = np.log1p(test_set["hours"])

regla_rating = {True: 1, False: 0}
train_set['rating'] = train_set['is_recommended'].map(regla_rating)
test_set['rating'] = test_set['is_recommended'].map(regla_rating)

ratings_ = test_set[test_set["rating"] == 1]
items_relevantes = test_set.groupby("user_id")["app_id"].apply(list).to_dict()


In [4]:
final_dict = {}
info_videojuegos = defaultdict(list)
set_tags = set()

with open(ruta_metadata, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        app_id = obj["app_id"]
        final_dict[app_id] = str(obj["description"])
        info_videojuegos[app_id].extend(obj["tags"])

        for tag in obj["tags"]:
            set_tags.add(tag)

# Lista total de app_id presentes en train y test
app_ids_total = train_set["app_id"].tolist()
app_ids_total.extend(test_set["app_id"].tolist())
app_ids_total = list(set(app_ids_total))
len(app_ids_total)


2880

In [5]:
#img_data = pd.read_csv(ruta_imagenes_url)
#img_data = img_data[["steam_appid", "header_image"]]
#img_data = img_data[img_data["steam_appid"].isin(app_ids_total)]
#img_data.head()


In [ ]:


# import requests
# from io import BytesIO
# from PIL import Image
# from sentence_transformers import SentenceTransformer
#
# model_img = SentenceTransformer("clip-ViT-B-32")
#
# embeddings_dict = {}
#
# for _, row in tqdm(img_data.iterrows(), total=len(img_data), desc="Procesando imágenes"):
#     image_id = int(row["steam_appid"])
#     url = row["header_image"]
#
#     try:
#         # Descargar imagen
#         response = requests.get(url, timeout=10)
#         response.raise_for_status()
#
#         # Abrir como PIL
#         img = Image.open(BytesIO(response.content)).convert("RGB")
#
#         # Embedding
#         emb = model_img.encode(img)
#
#         embeddings_dict[image_id] = emb
#
#     except Exception as e:
#         print(f"⚠️ Error con ID {image_id}, URL {url}: {e}")
#

# with open("embeddings_dict.pkl", "wb") as f:
#     pickle.dump(embeddings_dict, f)


In [ ]:
with open("embeddings_dict.pkl", "rb") as f:
    embeddings_dict = pickle.load(f)

embeddings_dict = {int(k): np.array(v) for k, v in embeddings_dict.items()}

train_set["app_id"] = train_set["app_id"].astype(int)
test_set["app_id"] = test_set["app_id"].astype(int)


In [8]:
descripciones = list(final_dict.values())
keys_app_id = list(final_dict.keys())

train_set = train_set.sort_values("user_id").reset_index(drop=True)
test_set = test_set.sort_values("user_id").reset_index(drop=True)

# El mejor en mi caso fue minilm
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings_text = model.encode(descripciones, show_progress_bar=True)

dict_transformados = {int(i): j for i, j in zip(keys_app_id, embeddings_text)}

train_set["descripciones"] = train_set["app_id"].map(dict_transformados)
test_set["descripciones"] = test_set["app_id"].map(dict_transformados)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1590 [00:00<?, ?it/s]

In [9]:
img_dim = next(iter(embeddings_dict.values())).shape[0]
zero_img = np.zeros(img_dim, dtype=np.float32)

train_set["emb_img"] = train_set["app_id"].map(embeddings_dict).apply(
    lambda x: x if isinstance(x, np.ndarray) else zero_img
)
test_set["emb_img"] = test_set["app_id"].map(embeddings_dict).apply(
    lambda x: x if isinstance(x, np.ndarray) else zero_img
)

columnas_importantes = ["user_id", "app_id", "hours", "descripciones", "emb_img"]
train_set = train_set[columnas_importantes]
test_set = test_set[columnas_importantes]

train_set.head()


,user_id,app_id,hours,descripciones,emb_img
0,731,322330,4.226834,"[-0.11883838, 0.04829872, -0.0025480525, -0.01...","[-0.1327441, 0.27538148, -0.24782176, 0.193519..."
1,731,433340,3.505557,"[-0.11883838, 0.04829872, -0.0025480525, -0.01...","[-0.5361488, 0.3972314, 0.09718392, 0.03582586..."
2,731,4700,6.528689,"[0.07425501, 0.027489644, 0.0194676, -0.025177...","[-0.64909875, 0.1372838, 0.23721811, 0.3455800..."
3,731,394360,6.003146,"[-0.11883838, 0.04829872, -0.0025480525, -0.01...","[-0.2154008, -0.29375905, -0.28681642, -0.2764..."
4,3128,1062090,0.182322,"[-0.11883838, 0.04829872, -0.0025480525, -0.01...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [ ]:
columnas_train, columnas_predict = ["user_id", "app_id", "descripciones", "emb_img"], ["hours"]
X_train_df, y_train = train_set[columnas_train], train_set[columnas_predict]

numerical_features_train = X_train_df[["user_id", "app_id"]].values

descripciones_dense_train = np.vstack(X_train_df["descripciones"].to_numpy())

img_emb_train = np.vstack(X_train_df["emb_img"].to_numpy())

X_train = np.hstack((numerical_features_train,
                     descripciones_dense_train,
                     img_emb_train))

group_train = train_set.groupby("user_id").size().tolist()
group_test = test_set.groupby("user_id").size().tolist() 

X_train.shape, len(group_train)


((63905, 898), 9906)

In [11]:
ranker = XGBRanker(
    objective="rank:pairwise",
    learning_rate=0.1,
    n_estimators=300,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

ranker.fit(
    X_train,
    y_train,
    group=group_train
)


XGBRanker(base_score=None, booster=None, callbacks=None, colsample_bylevel=None,
          colsample_bynode=None, colsample_bytree=0.8, device=None,
          early_stopping_rounds=None, enable_categorical=False,
          eval_metric=None, feature_types=None, feature_weights=None,
          gamma=None, grow_policy=None, importance_type=None,
          interaction_constraints=None, learning_rate=0.1, max_bin=None,
          max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=None,
          max_depth=6, max_leaves=None, min_child_weight=None, missing=nan,
          monotone_constraints=None, multi_strategy=None, n_estimators=300,
          n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
recomendaciones = defaultdict(list)

test_set_user_uniques = test_set["user_id"].unique().tolist()
test_set_app_uniques = test_set["app_id"].unique().tolist()
app_ids_array = np.array(list(test_set_app_uniques))

for user_id in tqdm(test_set_user_uniques, desc="Prediciendo por usuario"):
    app_ids = app_ids_array
    user_ids = np.full_like(app_ids, fill_value=user_id)

    numerical_feats = np.stack([user_ids, app_ids], axis=1)

    desc_feats = np.stack(
        [dict_transformados[int(app_id)] for app_id in app_ids]
    )

    img_feats = np.stack(
        [embeddings_dict.get(int(app_id), zero_img) for app_id in app_ids]
    )

    X_user = np.hstack((numerical_feats, desc_feats, img_feats))

    scores = ranker.predict(X_user)
    recomendaciones[user_id] = list(zip(app_ids, scores))

recomendaciones_def = defaultdict(list)
for usuario, lista_recs in recomendaciones.items():
    recomendaciones_ord = sorted(lista_recs, key=lambda x: x[1], reverse=True)
    recomendaciones_ord = [i[0] for i in recomendaciones_ord]
    recomendaciones_def[usuario] = recomendaciones_ord


Prediciendo por usuario: 100%|██████████| 9906/9906 [06:04<00:00, 27.15it/s]


In [14]:
precision_list = []
recall_list = []
ndcg_list = []
f1_list = []
hitrate_list = []
map10_list = []
diversity_list = []

for usuario, recomendaciones_usuario in recomendaciones_def.items():
    if usuario not in items_relevantes:
        continue

    items_rel_usuario = items_relevantes[usuario]
    recomendaciones_10 = recomendaciones_usuario[:10]

    precision_usuario = precision_at_k(recomendaciones_10, items_rel_usuario)
    recall_usuario = recall_at_k(recomendaciones_10, items_rel_usuario)
    ndcg_usuario = ndcg_at_k(recomendaciones_10, items_rel_usuario)
    f1_usuario = f1_at_k(recomendaciones_10, items_rel_usuario)
    hitrate_usuario = hit_score_at_k(recomendaciones_10, items_rel_usuario)
    map10_usuario = map_at_k(recomendaciones_10, items_rel_usuario)
    diversity_usuario = diversity_at_k(recomendaciones_10, info_videojuegos)

    precision_list.append(precision_usuario)
    recall_list.append(recall_usuario)
    ndcg_list.append(ndcg_usuario)
    f1_list.append(f1_usuario)
    hitrate_list.append(hitrate_usuario)
    map10_list.append(map10_usuario)
    diversity_list.append(diversity_usuario)

print(f"Precision@10: {np.mean(precision_list):.4f}")
print(f"Recall@10:    {np.mean(recall_list):.4f}")
print(f"nDCG@10:      {np.mean(ndcg_list):.4f}")
print(f"F1-Score@10:  {np.mean(f1_list):.4f}")
print(f"Hit Score@10: {np.mean(hitrate_list):.4f}")
print(f"MAP@10:       {np.mean(map10_list):.4f}")
print(f"Diversity:    {np.mean(diversity_list):.4f}")


Precision@10: 0.0020
Recall@10:    0.0167
nDCG@10:      0.0075
F1-Score@10:  0.0034
Hit Score@10: 0.0196
MAP@10:       0.0044
Diversity:    5.3813
